# Введение в PyTorch: От теории к практике

Этот ноутбук демонстрирует базовые строительные блоки PyTorch. Мы создадим набор данных, аналогичный тем, что используются в TensorFlow Playground, и обучим нейронную сеть для задачи бинарной классификации.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_circles

## 1. Подготовка данных (Аналог TF Playground)
Создадим нелинейно разделимый набор данных (вложенные круги).

In [ ]:
# Генерация данных
X_np, y_np = make_circles(n_samples=1000, noise=0.05, factor=0.5, random_state=42)

# Визуализация
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap=plt.cm.RdYlBu)
plt.title("Dataset: Circles")
plt.show()

# Конвертация в тензоры PyTorch (best practice: использовать float32 для весов и входов)
X = torch.FloatTensor(X_np)
y = torch.FloatTensor(y_np).unsqueeze(1) # Приводим к размерности [N, 1]

## 2. Dataset и DataLoader
Создание объектов для эффективной пакетной (batch) обработки данных.

In [ ]:
dataset = TensorDataset(X, y)
# shuffle=True важен для перемешивания данных перед каждой эпохой (помогает сходимости)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## 3. Архитектура модели
Определяем структуру нейронной сети, наследуясь от `nn.Module`. Важно: мы не используем сигмоиду или софтмакс на последнем слое. Сеть возвращает сырые логиты.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        # Определение слоев
        self.hidden1 = nn.Linear(2, 16)
        self.act1 = nn.ReLU()
        self.hidden2 = nn.Linear(16, 16)
        self.act2 = nn.ReLU()
        self.output = nn.Linear(16, 1) # Один выход для бинарной классификации (логит)
        
    def forward(self, x):
        x = self.act1(self.hidden1(x))
        x = self.act2(self.hidden2(x))
        x = self.output(x) # Возвращаем логиты
        return x

model = SimpleNet()

## 4. Логиты против Вероятностей (Softmax/Sigmoid)
Здесь мы используем `BCEWithLogitsLoss`. Эта функция объединяет слой Sigmoid и функцию потерь BCELoss в один класс. Это математически более стабильно, чем применение `nn.Sigmoid()` с последующим `nn.BCELoss()`, так как использует трюк Log-Sum-Exp. Обучение на логитах — это стандарт индустрии.

In [ ]:
criterion = nn.BCEWithLogitsLoss() # Ожидает логиты на входе
optimizer = optim.Adam(model.parameters(), lr=0.01) # Adam - стандартный оптимизатор по умолчанию

## 5. Цикл обучения (Training Loop)

In [ ]:
epochs = 100
loss_history = []

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        # 1. Forward pass: вычисление предсказаний (логитов)
        logits = model(batch_X)
        
        # 2. Вычисление потерь. criterion ожидает логиты благодаря BCEWithLogitsLoss
        loss = criterion(logits, batch_y)
        
        # 3. Очистка градиентов с предыдущего шага
        optimizer.zero_grad()
        
        # 4. Backward pass: обратное распространение ошибки
        loss.backward()
        
        # 5. Обновление весов
        optimizer.step()
        
        epoch_loss += loss.item()
        
    loss_history.append(epoch_loss / len(dataloader))
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss_history[-1]:.4f}')

## 6. Визуализация результатов
Построим кривую обучения и Decision Boundary, как в TensorFlow Playground.

In [ ]:
plt.plot(loss_history)
plt.title("Кривая обучения (Loss Curve)")
plt.xlabel("Эпохи")
plt.ylabel("Loss")
plt.show()

# Визуализация границы принятия решений
x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

grid = torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()])

# model.eval() отключает поведение слоев вроде Dropout и BatchNorm для инференса
model.eval()
with torch.no_grad(): # Отключаем расчет градиентов для ускорения
    logits_grid = model(grid)
    # Для получения вероятностей из логитов применяем сигмоиду
    probs = torch.sigmoid(logits_grid)
    # Переводим в классы (0 или 1)
    preds = (probs > 0.5).float()

Z = preds.numpy().reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.8, cmap=plt.cm.RdYlBu)
plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, edgecolors='k', cmap=plt.cm.RdYlBu)
plt.title("Decision Boundary")
plt.show()